In [3]:
PROJECT_ROOT = "/Users/bavithra/Documents/Uni/Courses/26_HTN/Qritical.jl/docs/"
using Pkg

Pkg.activate(PROJECT_ROOT)

  Activating project at `~/Documents/Uni/Courses/26_HTN/Qritical.jl/docs`


## MPS to LCF

In [8]:
"""
    left_canonical_mps(tensors; D=nothing, threshold=1e-12)

Bring an MPS given as a vector of site tensors into left canonical form by
sweeping SVDs from site 1 to site N-1.

Each output tensor `A[i]` satisfies the left-orthonormality condition:

    Σ_σ  A[i]†_{σ} A[i]_{σ}  =  I

# Arguments
- `tensors`: `Vector` of arrays. `tensors[1]` has shape `(d, k)`,
  `tensors[i]` for `1 < i < N` has shape `(k_l, d, k_r)`,
  `tensors[N]` has shape `(k, d)`.
- `D`: Maximum bond dimension. `nothing` means no truncation beyond threshold.
- `threshold`: Singular values below this are discarded. Defaults to `1e-12`.

# Returns
A `Vector` of left-canonical site tensors with the same index structure as
the input.
"""
function left_canonical_mps(tensors; D=nothing, threshold=1e-12)
    N = length(tensors)
    sites = deepcopy(tensors)

    for i in 1:N-1
        A = sites[i]

        # merge left bond + physical index into rows
        # site 1: shape (d, k_r)         → already (d, k_r), no reshape needed
        # site i: shape (k_l, d, k_r)    → reshape to (k_l*d, k_r)
        if ndims(A) == 2
            k_r = size(A, 2)
            M = A                          # (d, k_r)
        else
            k_l, d, k_r = size(A)
            M = reshape(A, k_l * d, k_r)  # (k_l*d, k_r)
        end

        U, Σ, Vt = factorize_with_svd(M; discard_below_threshold=true, threshold=threshold)

        # cap bond dimension
        if !isnothing(D)
            k = min(D, size(U, 2))
            U, Σ, Vt = U[:, 1:k], Σ[1:k, 1:k], Vt[1:k, :]
        end

        k = size(U, 2)

        # store left-canonical site tensor — restore original index structure
        if ndims(A) == 2
            sites[i] = U                              # (d, k)
        else
            sites[i] = reshape(U, k_l, d, k)         # (k_l, d, k)
        end

        # absorb Σ Vt into the next site
        ΣVt = Σ * Vt                                  # (k, k_r)
        B = sites[i+1]

        if i + 1 == N
            # last site: shape (k_r, d) → contract on left index
            sites[i+1] = ΣVt * reshape(B, k_r, size(B, 2))
        else
            k_l2, d2, k_r2 = size(B)
            # reshape next site to (k_l2, d2*k_r2), left-multiply, reshape back
            sites[i+1] = reshape(ΣVt * reshape(B, k_l2, d2 * k_r2), k, d2, k_r2)
        end
    end

    return sites
end

left_canonical_mps

## MPS to RCF

In [9]:
"""
    right_canonical_mps(tensors; D=nothing, threshold=1e-12)

Bring an MPS into right canonical form by sweeping SVDs from site N to site 2.

Each output tensor `B[i]` satisfies the right-orthonormality condition:

    Σ_σ  B[i]_{σ} B[i]†_{σ}  =  I

# Arguments
- `tensors`: `Vector` of site tensors, same shape convention as
  `left_canonical_mps`.
- `D`: Maximum bond dimension. `nothing` means no truncation beyond threshold.
- `threshold`: Singular values below this are discarded. Defaults to `1e-12`.

# Returns
A `Vector` of right-canonical site tensors.
"""
function right_canonical_mps(tensors; D=nothing, threshold=1e-12)
    N = length(tensors)
    sites = deepcopy(tensors)

    for i in N:-1:2
        B = sites[i]

        # merge physical index + right bond into columns
        # last site: shape (k_l, d)         → already (k_l, d)
        # site i:    shape (k_l, d, k_r)    → reshape to (k_l, d*k_r)
        if ndims(B) == 2
            k_l = size(B, 1)
            M = B                            # (k_l, d)
        else
            k_l, d, k_r = size(B)
            M = reshape(B, k_l, d * k_r)    # (k_l, d*k_r)
        end

        U, Σ, Vt = factorize_with_svd(M; discard_below_threshold=true, threshold=threshold)

        if !isnothing(D)
            k = min(D, size(Vt, 1))
            U, Σ, Vt = U[:, 1:k], Σ[1:k, 1:k], Vt[1:k, :]
        end

        k = size(Vt, 1)

        # store right-canonical site tensor
        if ndims(B) == 2
            sites[i] = Vt                              # (k, d)
        else
            sites[i] = reshape(Vt, k, d, k_r)         # (k, d, k_r)
        end

        # absorb U Σ into the previous site
        UΣ = U * Σ                                     # (k_l, k)
        A = sites[i-1]

        if i - 1 == 1
            # first site: shape (d, k_l) → contract on right index
            sites[i-1] = reshape(A, size(A, 1), k_l) * UΣ
        else
            k_l2, d2, k_r2 = size(A)
            sites[i-1] = reshape(reshape(A, k_l2 * d2, k_r2) * UΣ, k_l2, d2, k)
        end
    end

    return sites
end

right_canonical_mps

In [ ]:
using Serialization
mps = deserialize("psi.jls")         # or output of left_canonical() from Ex1

# bring to left canonical
mps_L = left_canonical_mps(mps; D=16, threshold=1e-10)

# bring to right canonical
mps_R = right_canonical_mps(mps; D=16, threshold=1e-10)

# should be ~1e-12

BoundsError: BoundsError: attempt to access Tuple{} at index [1]

In [ ]:
# sanity check left-orthonormality at site 3
A = mps_L[3]
Am = reshape(A, size(A, 1) * size(A, 2), size(A, 3))   # merge (k_l, d) into rows
@info "A†A ≈ I?" norm(Am' * Am - I)

- to check if the different A forms are correct the sanity checks - normalization conditions, observable values
- throwing away values 10-7,-8,-9 are good values
- friedel oscillations

In [ ]:
# looking at the entanglement spectrum to check the area and volume law states

In [ ]:
mutable struct MPS{T<:Number}
    tensors::Vector{Array{T,3}}
    discardedsq::Float64
end

mutable struct BondaCanonicalMPS{T<:Number}


In [ ]:
using TensorOperations